# Seamless Interaction Dataset Download and Preprocessing

This notebook downloads and prepares the Improvised test split of the Seamless Interaction dataset used throughout the conversational anomaly detection experiments.

The objective is to construct a persistent, participant-centric audiovisual dataset containing the source video, audio, and metadata required by the subsequent experimental pipelines.

The notebook performs the following steps:

Connects Google Colab to Google Drive and prepares persistent storage.
Discovers all available files in the facebook/seamless-interaction Hugging Face repository.
Selects all archive shards belonging to improvised/test.
Downloads the shards sequentially with checkpoint-based resume support.
Retains participant-specific:
video (.mp4),
audio (.wav),
metadata (.json).
Preprocesses each video by:
retaining the first 120 seconds,
cropping the upper half of the portrait frame,
resizing the cropped video to a height of 720 pixels,
re-encoding the result with H.264.
Discards unused .npz representations.
Organizes the retained files by conversation and participant.
Performs structural integrity checks and removes incomplete conversation folders.
Separates participant records with empty transcript metadata into a dedicated silent/ pool.
Performs final validation of the resulting dataset structure.
Optionally creates a compressed final_data.zip archive in Google Drive.

The resulting processed dataset is used as the common source for all later participant-centric semantic, temporal, participation, consolidation, ablation, and fine-tuning experiments.

**Scope:** This notebook performs dataset acquisition and low-level preprocessing only. It does not construct Normal, Wrong Partner, Lag Partner, or Silent Partner experimental cases, and it does not apply the later experiment-specific eligibility criteria used to select the final evaluation pools.


**1: Environment and Persistent Storage**

The preprocessing pipeline is designed to run in Google Colab while storing all persistent outputs in Google Drive.

The required Python package for Hugging Face dataset access is installed together with FFmpeg, which is used for video trimming, cropping, resizing, and re-encoding.

In [ ]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

!pip -q install huggingface_hub
!apt-get -qq update
!apt-get -qq install -y ffmpeg


Mounted at /content/drive
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


**2: Directory Configuration**
A persistent working directory is created under:

/content/drive/MyDrive/seamless_download/

Processed dataset files and download checkpoints are stored in Google Drive so that preprocessing can resume after a Colab runtime interruption.

Temporary shard and raw-video files are instead stored in the local Colab filesystem and deleted after processing to avoid unnecessarily consuming Google Drive storage.

In [ ]:
from pathlib import Path
import os
import json
import tarfile
import shutil
import subprocess
from collections import Counter
from huggingface_hub import list_repo_files, hf_hub_download

repo_id = "facebook/seamless-interaction"

# Persistent working directory in Google Drive
WORK_DIR = Path("/content/drive/MyDrive/seamless_download")

# Persistent outputs/checkpoints
data_dir = WORK_DIR / "data"
hf_cache_dir = WORK_DIR / "hf_cache"
drive_output_dir = WORK_DIR

# Local temporary folders: faster and safe to delete
tmp_dir = Path("/content/tmp_videos")
tmp_shard_dir = Path("/content/tmp_shards")

for d in [WORK_DIR, data_dir, hf_cache_dir, drive_output_dir, tmp_dir, tmp_shard_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Make HuggingFace cache persistent
os.environ["HF_HOME"] = str(hf_cache_dir)
os.environ["HF_HUB_CACHE"] = str(hf_cache_dir / "hub")

print("Data dir:", data_dir)
print("HF cache dir:", hf_cache_dir)
print("Temp video dir:", tmp_dir)
print("Temp shard dir:", tmp_shard_dir)
print("Drive output dir:", drive_output_dir)


Data dir: /content/drive/MyDrive/seamless_download/data
HF cache dir: /content/drive/MyDrive/seamless_download/hf_cache
Temp video dir: /content/tmp_videos
Temp shard dir: /content/tmp_shards
Drive output dir: /content/drive/MyDrive/seamless_download


**3: Dataset Repository Discovery**

The Hugging Face repository is queried to obtain the complete list of available Seamless Interaction files.

Only files belonging to the Improvised test split are used by this project. The following cells identify all .tar and .tar.gz archives under improvised/test, including the associated audiovisual and metadata shards required by the preprocessing pipeline.

In [ ]:
files = list_repo_files(repo_id, repo_type="dataset")

print("Number of files:", len(files))
for f in files[:100]:
    print(f)


Number of files: 21970
.gitattributes
LICENSE
README.md
improvised/dev/0000/0000.tar
improvised/dev/0000/0001.tar
improvised/dev/0000/0002.tar
improvised/dev/0000/0003.tar
improvised/dev/0000/0004.tar
improvised/dev/0000/0005.tar
improvised/dev/0000/0006.tar
improvised/dev/0000/0007.tar
improvised/dev/0000/0008.tar
improvised/dev/0000/0009.tar
improvised/dev/0000/0010.tar
improvised/dev/0000/0011.tar
improvised/dev/0000/0012.tar
improvised/dev/0000/0013.tar
improvised/dev/0000/0014.tar
improvised/dev/0000/0015.tar
improvised/dev/0000/0016.tar
improvised/dev/0000/0017.tar
improvised/dev/0000/0018.tar
improvised/dev/0000/0019.tar
improvised/dev/0000/0020.tar
improvised/dev/0000/0021.tar
improvised/dev/0000/0022.tar
improvised/dev/0000/0023.tar
improvised/dev/0000/0024.tar
improvised/dev/0000/0025.tar
improvised/dev/0000/0026.tar
improvised/dev/0000/0027.tar
improvised/dev/0000/0028.tar
improvised/dev/0000/0029.tar
improvised/dev/0000/0030.tar
improvised/dev/0000/0031.tar
improvised/dev/0

In [ ]:
test_files = sorted([
    f for f in files
    if f.startswith("improvised/test") and (f.endswith(".tar") or f.endswith(".tar.gz"))
])

print("Number of test shards:", len(test_files))
print("First 10 test shards:")
for f in test_files[:10]:
    print(f)
print("Last 10 test shards:")
for f in test_files[-10:]:
    print(f)


Number of test shards: 83
First 10 test shards:
improvised/test/0000/0000.tar
improvised/test/0000/0001.tar
improvised/test/0000/0002.tar
improvised/test/0000/0003.tar
improvised/test/0000/0004.tar
improvised/test/0000/0005.tar
improvised/test/0000/0006.tar
improvised/test/0000/0007.tar
improvised/test/0000/0008.tar
improvised/test/0000/0009.tar
Last 10 test shards:
improvised/test/0000/0073.tar
improvised/test/0000/0074.tar
improvised/test/0000/0075.tar
improvised/test/0000/0076.tar
improvised/test/0000/0077.tar
improvised/test/0000/0078.tar
improvised/test/extras/audio/0000.tar
improvised/test/extras/audio/0001.tar
improvised/test/extras/metadata/transcript/0000.tar
improvised/test/extras/metadata/vad/0000.tar


**4: Resumable Shard Processing**

Downloading and preprocessing the complete test split is computationally and storage intensive. To make the procedure robust to Colab runtime interruptions, processed shard names are recorded in:

processed_test_shards.txt

When the notebook is restarted, previously completed shards are skipped automatically.

Setting START_FROM_SCRATCH = True removes the existing processed dataset and checkpoint log and reconstructs the dataset from the beginning. For normal interrupted-session recovery, it should remain False.


In [ ]:
# If True, deletes Drive data/checkpoint and starts from scratch.
# Keep False if you want to resume after a Colab interruption/runtime reset.
START_FROM_SCRATCH = False

processed_shards_log = WORK_DIR / "processed_test_shards.txt"

if START_FROM_SCRATCH:
    if data_dir.exists():
        shutil.rmtree(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    if processed_shards_log.exists():
        processed_shards_log.unlink()

processed_shards = set()
if processed_shards_log.exists():
    processed_shards = set(processed_shards_log.read_text().splitlines())

print("Already processed shards:", len(processed_shards))
print("Total test shards to process:", len(test_files))


Already processed shards: 83
Total test shards to process: 83


**5: Download and Audiovisual Preprocessing**

Each selected test shard is downloaded and processed independently.

For every archive:

.mp4 participant videos are temporarily extracted and processed with FFmpeg;
only the first 120 seconds of each video are retained;
the upper half of each portrait-oriented frame is cropped to preserve primarily facial and upper-body behaviour;
the cropped frame is resized to a height of 720 pixels while preserving its aspect ratio;
the video is re-encoded using H.264;
participant-specific .wav audio files are retained without additional transformation;
.json metadata files are retained;
.npz representations and other unused files are ignored.

After a shard has been processed successfully, the downloaded archive is deleted from local storage and its name is added to the persistent checkpoint file.

This produces the audiovisual and metadata representation used by the downstream experiments while substantially reducing the storage and inference cost of the original videos.

In [ ]:
for shard_idx, shard in enumerate(test_files, start=1):
    if shard in processed_shards:
        print(f"=== Skipping already processed shard {shard_idx}/{len(test_files)}: {shard} ===")
        continue

    print(f"=== Downloading shard {shard_idx}/{len(test_files)}: {shard} ===")

    shard_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=shard,
        local_dir=str(tmp_shard_dir),
        local_dir_use_symlinks=False,
    )

    try:
        with tarfile.open(shard_path) as tar:
            for member in tar.getmembers():
                filename = Path(member.name).name

                # -----------------------------
                # Process videos
                # -----------------------------
                if filename.endswith(".mp4"):
                    print("Processing video:", filename)

                    temp_input = tmp_dir / filename
                    output_video = data_dir / filename

                    try:
                        # Extract raw mp4 temporarily
                        with tar.extractfile(member) as src, open(temp_input, "wb") as dst:
                            dst.write(src.read())

                        # FFmpeg preprocessing:
                        # - keep first 120 sec
                        # - crop upper half
                        # - resize to 720 height
                        cmd = [
                            "ffmpeg",
                            "-y",
                            "-i", str(temp_input),
                            "-t", "120",
                            "-vf", "crop=in_w:in_h/2:0:0,scale=-1:720",
                            "-c:v", "libx264",
                            "-preset", "veryfast",
                            "-crf", "23",
                            "-movflags", "+faststart",
                            str(output_video),
                        ]

                        result = subprocess.run(
                            cmd,
                            stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE,
                            text=True,
                        )

                        if result.returncode != 0:
                            print("FFMPEG FAILED for", filename)
                            print(result.stderr)
                            if output_video.exists():
                                output_video.unlink()
                        else:
                            print("Saved:", output_video)

                    finally:
                        # Delete temporary raw video
                        if temp_input.exists():
                            temp_input.unlink()

                # -----------------------------
                # Keep JSON and WAV only
                # -----------------------------
                elif filename.endswith(".json") or filename.endswith(".wav"):
                    print("Saving metadata/audio:", filename)

                    with tar.extractfile(member) as src, open(data_dir / filename, "wb") as dst:
                        dst.write(src.read())

                # -----------------------------
                # Ignore NPZ for now
                # -----------------------------
                else:
                    continue

        with open(processed_shards_log, "a", encoding="utf-8") as f:
            f.write(shard + "\n")
        processed_shards.add(shard)

    finally:
        try:
            if os.path.exists(shard_path):
                os.remove(shard_path)
                print("Deleted local shard:", shard_path)
        except Exception as e:
            print("Could not delete local shard:", e)

print("ALL TEST SHARDS PROCESSED")


=== Skipping already processed shard 1/83: improvised/test/0000/0000.tar ===
=== Skipping already processed shard 2/83: improvised/test/0000/0001.tar ===
=== Skipping already processed shard 3/83: improvised/test/0000/0002.tar ===
=== Skipping already processed shard 4/83: improvised/test/0000/0003.tar ===
=== Skipping already processed shard 5/83: improvised/test/0000/0004.tar ===
=== Skipping already processed shard 6/83: improvised/test/0000/0005.tar ===
=== Skipping already processed shard 7/83: improvised/test/0000/0006.tar ===
=== Skipping already processed shard 8/83: improvised/test/0000/0007.tar ===
=== Skipping already processed shard 9/83: improvised/test/0000/0008.tar ===
=== Skipping already processed shard 10/83: improvised/test/0000/0009.tar ===
=== Skipping already processed shard 11/83: improvised/test/0000/0010.tar ===
=== Skipping already processed shard 12/83: improvised/test/0000/0011.tar ===
=== Skipping already processed shard 13/83: improvised/test/0000/0012.tar

**6: Organize Files by Conversation and Participant**

The extracted files are initially stored in a flat directory. They are reorganized using the identifiers encoded in their filenames.

The resulting participant-centric structure is:

```text
data/
├── <conversation_id>/
│   ├── <participant_A>/
│   │   ├── video / audio / metadata files
│   │   └── ...
│   └── <participant_B>/
│       ├── video / audio / metadata files
│       └── ...
└── ...
```

This representation makes each participant stream independently addressable while preserving the original conversation-level pairing, which is required by the participant-centric experimental framework.

In [ ]:
files = list(data_dir.glob("*"))

for f in files:
    if f.is_dir():
        continue

    name = f.stem
    parts = name.split("_")

    if len(parts) < 4:
        continue

    conversation_id = "_".join(parts[:3])
    participant_id = parts[3]

    conv_dir = data_dir / conversation_id
    part_dir = conv_dir / participant_id

    part_dir.mkdir(parents=True, exist_ok=True)

    new_path = part_dir / f.name
    shutil.move(str(f), str(new_path))

print("Dataset reorganized by conversation and participant.")


Dataset reorganized by conversation and participant.


**7: Initial Structural Integrity Check**

Before constructing the experimental dataset, the organized directory structure is inspected to determine how many participant folders are associated with each conversation identifier.

A valid dyadic source conversation is expected to contain exactly two participant directories.

This check exposes incomplete or structurally irregular records before they can enter the downstream experiments.


In [ ]:
conversations = [d for d in data_dir.iterdir() if d.is_dir()]

stats = Counter()
for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]
    stats[len(participants)] += 1

print("Total conversations:", len(conversations))
print()
for k in sorted(stats):
    print(f"{k} participants:", stats[k])


Total conversations: 233

0 participants: 23
1 participants: 3
2 participants: 206
26 participants: 1


**8: Identify Dataset-Native Silent Participant Records**

Participant metadata are inspected for records with empty transcript information.

Participants with no transcript content are separated from the ordinary dyadic conversation pool and moved to a dedicated:

data/silent/

directory while retaining their original conversation and participant identifiers.

These records are preserved rather than discarded because they are later used as a dataset-native source of absent spoken participation. No artificial muting or modification of the corresponding audiovisual stream is performed at this stage.

In [ ]:
from tqdm.auto import tqdm

silent_root = data_dir / "silent"
silent_root.mkdir(exist_ok=True)

conversations = [
    d for d in data_dir.iterdir()
    if d.is_dir() and d.name != "silent"
]

moved_count = 0
checked_count = 0
errors = []

for conv in tqdm(conversations, desc="Checking conversations"):
    participants = [p for p in conv.iterdir() if p.is_dir()]

    # Look only at conversations with 2 participants
    if len(participants) != 2:
        continue

    for participant_dir in participants:
        checked_count += 1

        json_files = list(participant_dir.glob("*.json"))
        if len(json_files) != 1:
            errors.append(f"Expected 1 json in {participant_dir}, found {len(json_files)}")
            continue

        json_path = json_files[0]

        try:
            with open(json_path, "r", encoding="utf-8") as f:
                sample_json = json.load(f)

            transcript = sample_json.get("metadata:transcript", None)

            # silent if [] or "" or None or generally empty
            is_silent = not transcript

            if is_silent:
                target_dir = silent_root / conv.name / participant_dir.name
                target_dir.parent.mkdir(parents=True, exist_ok=True)

                print(f"Moving silent participant: {participant_dir} -> {target_dir}")
                shutil.move(str(participant_dir), str(target_dir))
                moved_count += 1

        except Exception as e:
            errors.append(f"{participant_dir}: {e}")

print("=== DONE ===")
print("Participants checked:", checked_count)
print("Silent participants moved:", moved_count)

if errors:
    print("Errors:")
    for err in errors:
        print("-", err)

Checking conversations:   0%|          | 0/232 [00:00<?, ?it/s]

=== DONE ===
Participants checked: 412
Silent participants moved: 0


In [ ]:
conversations = [d for d in data_dir.iterdir() if d.is_dir()]

stats = Counter()
for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]
    stats[len(participants)] += 1

print("Total conversations:", len(conversations))
print()
for k in sorted(stats):
    print(f"{k} participants:", stats[k])


Total conversations: 233

0 participants: 23
1 participants: 3
2 participants: 206
26 participants: 1


In [ ]:
conversations = [
    d for d in data_dir.iterdir()
    if d.is_dir() and d.name != "silent"
]

checked_conversations = 0
empty_transcripts = []

for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]

    # only conversations with 2 participants
    if len(participants) != 2:
        continue

    checked_conversations += 1

    for participant_dir in participants:
        json_files = list(participant_dir.glob("*.json"))
        if len(json_files) != 1:
            continue

        json_path = json_files[0]

        with open(json_path, "r", encoding="utf-8") as f:
            sample_json = json.load(f)

        transcript = sample_json.get("metadata:transcript")

        if transcript == [] or transcript == "":
            empty_transcripts.append((conv.name, participant_dir.name))

print("Checked conversations:", checked_conversations)
print("Participants with empty transcript:", len(empty_transcripts))
print()
for item in empty_transcripts[:20]:
    print(item)


Checked conversations: 206
Participants with empty transcript: 0



**9: Remove Incomplete Conversation Records**

After separating the silent-participant pool, the remaining source conversations are cleaned so that the main dataset contains only structurally complete dyadic interactions.

Conversation directories that do not contain exactly two participant streams are removed from the main conversation pool.

The `silent/` directory is excluded from this operation because it is maintained separately for later participation experiments.

In [ ]:
import shutil

deleted = 0

invalid_counts = {0, 1, 26}

for conv in data_dir.iterdir():
    if not conv.is_dir():
        continue

    if conv.name == "silent":
        continue

    participants = [p for p in conv.iterdir() if p.is_dir()]

    if len(participants) in invalid_counts:
        print(f"Deleting ({len(participants)} participants):", conv)
        shutil.rmtree(conv)
        deleted += 1

print("\nDeleted conversations:", deleted)

Deleting (0 participants): /content/drive/MyDrive/seamless_download/data/V01_S0173_I00001224
Deleting (1 participants): /content/drive/MyDrive/seamless_download/data/V01_S1545_I00000129
Deleting (1 participants): /content/drive/MyDrive/seamless_download/data/V01_S1545_I00000138
Deleting (0 participants): /content/drive/MyDrive/seamless_download/data/V01_S0173_I00001232
Deleting (0 participants): /content/drive/MyDrive/seamless_download/data/V01_S0173_I00001233
Deleting (0 participants): /content/drive/MyDrive/seamless_download/data/V01_S0173_I00001227
Deleting (0 participants): /content/drive/MyDrive/seamless_download/data/V03_S1088_I00000498
Deleting (0 participants): /content/drive/MyDrive/seamless_download/data/V01_S0307_I00001225
Deleting (0 participants): /content/drive/MyDrive/seamless_download/data/V01_S0307_I00001226
Deleting (1 participants): /content/drive/MyDrive/seamless_download/data/V01_S0337_I00001104
Deleting (0 participants): /content/drive/MyDrive/seamless_download/da

**10: Post-Cleaning Validation**

The cleaned dataset is validated again to ensure that:

- every retained ordinary conversation contains exactly two participant directories;
- both participants contain the expected metadata;
- no retained participant has an empty transcript;
- the silent-participant records remain isolated under the dedicated `silent/` directory.

The following checks verify the final conversation counts and confirm the separation between the complete dyadic source pool and the silent-participant pool.

In [ ]:
# Check again conversations with 2 participants
conversations = [
    d for d in data_dir.iterdir()
    if d.is_dir() and d.name != "silent"
]

checked_conversations = 0
empty_transcripts = []

for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]

    # only conversations with 2 participants
    if len(participants) != 2:
        continue

    checked_conversations += 1

    for participant_dir in participants:
        json_files = list(participant_dir.glob("*.json"))
        if len(json_files) != 1:
            continue

        json_path = json_files[0]

        with open(json_path, "r", encoding="utf-8") as f:
            sample_json = json.load(f)

        transcript = sample_json.get("metadata:transcript")

        if transcript == [] or transcript == "":
            empty_transcripts.append((conv.name, participant_dir.name))

print("Checked conversations:", checked_conversations)
print("Participants with empty transcript:", len(empty_transcripts))
print()
for item in empty_transcripts[:20]:
    print(item)


Checked conversations: 206
Participants with empty transcript: 0



In [ ]:
conversations = [d for d in data_dir.iterdir() if d.is_dir()]

stats = Counter()
for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]
    stats[len(participants)] += 1

print("Total conversations:", len(conversations))
print()
for k in sorted(stats):
    print(f"{k} participants:", stats[k])


Total conversations: 207

2 participants: 206
26 participants: 1


In [ ]:
for conv in data_dir.iterdir():
    if conv.is_dir():
        participants = [p for p in conv.iterdir() if p.is_dir()]
        if len(participants) != 2:
            print(conv.name, len(participants))

silent 26


In [ ]:
from collections import Counter

conversations = [
    d for d in data_dir.iterdir()
    if d.is_dir() and d.name != "silent"
]

stats = Counter()

for conv in conversations:
    participants = [p for p in conv.iterdir() if p.is_dir()]
    stats[len(participants)] += 1

print(f"Normal conversations: {len(conversations)}")
print(stats)

silent_convs = [
    d for d in (data_dir / "silent").iterdir()
    if d.is_dir()
]

print(f"Silent conversations: {len(silent_convs)}")

Normal conversations: 206
Counter({2: 206})
Silent conversations: 26


**11: Final Dataset Validation**

A final inspection reports the top-level dataset structure and disk usage.

At this stage, the processed source dataset contains the complete dyadic conversations that passed the structural integrity checks together with the separately preserved silent-participant pool.

Experiment-specific eligibility filtering, source-level dataset splits, and anomaly construction are intentionally performed in later stages of the project rather than in this preprocessing notebook.

In [ ]:
print("Top-level folders inside data:")
for p in sorted(data_dir.iterdir())[:30]:
    print(p.name)

print("Disk usage:")
!du -sh "$data_dir"
!find "$data_dir" -maxdepth 3 -type f | head -20


**12: Optional Dataset Archive**

For convenient storage and transfer, the complete processed data/ directory can be compressed into:

seamless_download/final_data.zip

The archive contains the contents of the organized dataset directly and is stored persistently in Google Drive.

Creating the ZIP archive is optional for downstream experimentation: the uncompressed data/ directory can be used directly when it is already available in Google Drive.

In [ ]:
# Create final_data.zip from the organized data folder
zip_base = WORK_DIR / "final_data"

archive_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(data_dir)
)

size_gb = os.path.getsize(archive_path) / (1024 ** 3)

print("Created archive:", archive_path)
print(f"Final ZIP size: {size_gb:.2f} GB")

!ls -lh "$archive_path"


Created archive: /content/drive/MyDrive/seamless_download/final_data.zip
Final ZIP size: 10.37 GB
-rw------- 1 root root 11G Jul  9 11:02 /content/drive/MyDrive/seamless_download/final_data.zip
